# Sales Growth Analysis

End-to-end pipeline: load raw sales data, clean and transform it with Pandas, then load the cleaned dataset into a MySQL database for further analysis and reporting (Power BI dashboard).

## 1. Setup

Install dependencies and import libraries.

In [ ]:
# Install required packages (skip if already installed)
!pip install pandas pymysql sqlalchemy python-dotenv --quiet


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine


## 2. Load Raw Data

Place `sales_raw_500.csv` in a `data/` folder next to this notebook, or update `RAW_DATA_PATH` below to point to your file.

In [ ]:
RAW_DATA_PATH = os.path.join("data", "sales_raw_500.csv")

df = pd.read_csv(RAW_DATA_PATH)
df.head()


## 3. Clean the Data

- Remove duplicate rows
- Fix the date column and drop rows with invalid dates

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()


In [ ]:
# Parse Order_Date; invalid/mixed formats become NaT
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="mixed", errors="coerce")

# Drop rows where the date couldn't be parsed
df = df.dropna(subset=["Order_Date"])


## 4. Feature Engineering

Derive `Profit`, `Year`, and `Month` from the existing columns.

In [ ]:
df["Profit"] = df["Sales"] - df["Cost"]
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month

df.head()


## 5. Export Cleaned Data

Save the cleaned dataset to CSV for reuse (e.g. in Power BI).

In [ ]:
OUTPUT_PATH = os.path.join("data", "sales_cleaned.csv")
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned data saved to {OUTPUT_PATH}")


## 6. Load into MySQL

Database credentials are read from environment variables — **never hardcode credentials in a notebook.**

Set these before running (e.g. in a `.env` file loaded with `python-dotenv`, or your shell):

```
DB_USER=your_username
DB_PASSWORD=your_password
DB_HOST=localhost
DB_PORT=3306
DB_NAME=salesdb
```

In [ ]:
from dotenv import load_dotenv
from urllib.parse import quote_plus

load_dotenv()  # reads variables from a local .env file, if present

db_user = os.environ["DB_USER"]
db_password = quote_plus(os.environ["DB_PASSWORD"])  # safely encode special characters
db_host = os.environ.get("DB_HOST", "localhost")
db_port = os.environ.get("DB_PORT", "3306")
db_name = os.environ.get("DB_NAME", "salesdb")

connection_string = f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(connection_string)


In [ ]:
df.to_sql(name="sales", con=engine, if_exists="replace", index=False)
print("Data loaded into the 'sales' table successfully.")


## Next Steps

- Open `sales_Growth_Analysis.pbix` in Power BI and connect it to this MySQL table (or the cleaned CSV) to refresh the dashboard.
- See the project README for setup instructions.